# Phase 3 Experiments - Phase C+D: Pruning KD Models

This notebook runs:
- Run 3.3-3.6: Prune KD1-4 with Magnitude (4 runs)
- Run 4.1-4.4: Prune KD1-4 with Wanda (4 runs)
- Run 4.5-4.8: Prune KD1-4 with Gradual (4 runs)
- Run 4.9-4.12: Prune KD1-4 with Structured (4 runs)

**KEY OPTIMIZATION**: Uses `prune_only` pipeline with uploaded KD models from Phase B!
This avoids redoing KD for each pruning run, saving ~4 hours per experiment.

**IMPORTANT**: KD models use teacher's eval_fold for fair comparison!
- KD1, KD2 (Teacher T1): eval_fold=3
- KD3, KD4 (Teacher T2): eval_fold=2

**Estimated Time**: 6-8 hours (vs 20+ hours if redoing KD)

**DEPENDENCY**: Phase B must complete first and upload KD models to HuggingFace!

**Total Runs**: 16

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q iterative-stratification scikit-learn pandas numpy tqdm

In [ ]:
# Setup logging
import sys
from datetime import datetime

class Logger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w")
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger("/kaggle/working/experiment_log.txt")
print(f"Experiment started at: {datetime.now()}")
print("Phase C+D: Pruning KD Models (using uploaded models from Phase B)")

In [ ]:
# Clone repository
!git clone https://github.com/SaifSiddique009/kd_pruning_quantization_framework_for_nlp.git
%cd kd_pruning_quantization_framework_for_nlp
!git checkout phase3-comprehensive-experiments
!git log -1 --oneline

In [ ]:
# KD Model paths from Phase B (uploaded to HuggingFace)
AUTHOR = "Saif-Siddique"

KD_MODELS = {
    "KD1": f"{AUTHOR}/bangla-cyberbully-kd1-xlmroberta-to-sahajbert",
    "KD2": f"{AUTHOR}/bangla-cyberbully-kd2-xlmroberta-to-banglabert-small",
    "KD3": f"{AUTHOR}/bangla-cyberbully-kd3-banglabert-to-sahajbert",
    "KD4": f"{AUTHOR}/bangla-cyberbully-kd4-banglabert-to-banglabert-small",
}

print("KD Models to be used (from HuggingFace):")
for kd_id, path in KD_MODELS.items():
    print(f"  {kd_id}: {path}")

In [ ]:
# Verify KD models are accessible
from transformers import AutoModel

print("Verifying KD models are accessible...")
for kd_id, path in KD_MODELS.items():
    try:
        _ = AutoModel.from_pretrained(path)
        print(f"  [OK] {kd_id}: {path}")
        del _
    except Exception as e:
        print(f"  [ERROR] {kd_id}: {e}")
        print(f"  Make sure Phase B completed and uploaded models!")

---
## Scenario 3: Prune KD Models with Magnitude (Runs 3.3-3.6)

Uses `prune_only` pipeline with uploaded KD models!

---

### Run 3.3: KD1 + Magnitude Pruning (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd1-xlmroberta-to-sahajbert" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario3/KD1_magnitude

### Run 3.4: KD2 + Magnitude Pruning (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd2-xlmroberta-to-banglabert-small" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario3/KD2_magnitude

### Run 3.5: KD3 + Magnitude Pruning (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd3-banglabert-to-sahajbert" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario3/KD3_magnitude

### Run 3.6: KD4 + Magnitude Pruning (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd4-banglabert-to-banglabert-small" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario3/KD4_magnitude

---
## Scenario 4: Pruning Methods Comparison (Runs 4.1-4.12)

Compare Wanda, Gradual, and Structured pruning on all 4 KD models.

---

### Wanda Pruning (Runs 4.1-4.4)

#### Run 4.1: KD1 + Wanda (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd1-xlmroberta-to-sahajbert" \
    --prune_method wanda \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario4/KD1_wanda

#### Run 4.2: KD2 + Wanda (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd2-xlmroberta-to-banglabert-small" \
    --prune_method wanda \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario4/KD2_wanda

#### Run 4.3: KD3 + Wanda (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd3-banglabert-to-sahajbert" \
    --prune_method wanda \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario4/KD3_wanda

#### Run 4.4: KD4 + Wanda (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd4-banglabert-to-banglabert-small" \
    --prune_method wanda \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario4/KD4_wanda

### Gradual Pruning (Runs 4.5-4.8)

#### Run 4.5: KD1 + Gradual (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd1-xlmroberta-to-sahajbert" \
    --prune_method gradual \
    --prune_sparsity 0.5 \
    --prune_schedule cubic \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario4/KD1_gradual

#### Run 4.6: KD2 + Gradual (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd2-xlmroberta-to-banglabert-small" \
    --prune_method gradual \
    --prune_sparsity 0.5 \
    --prune_schedule cubic \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario4/KD2_gradual

#### Run 4.7: KD3 + Gradual (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd3-banglabert-to-sahajbert" \
    --prune_method gradual \
    --prune_sparsity 0.5 \
    --prune_schedule cubic \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario4/KD3_gradual

#### Run 4.8: KD4 + Gradual (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd4-banglabert-to-banglabert-small" \
    --prune_method gradual \
    --prune_sparsity 0.5 \
    --prune_schedule cubic \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario4/KD4_gradual

### Structured Pruning (Runs 4.9-4.12) - ACTUALLY REDUCES PARAMETERS!

#### Run 4.9: KD1 + Structured (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd1-xlmroberta-to-sahajbert" \
    --prune_method structured \
    --prune_sparsity 0.3 \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario4/KD1_structured

#### Run 4.10: KD2 + Structured (eval_fold=3)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd2-xlmroberta-to-banglabert-small" \
    --prune_method structured \
    --prune_sparsity 0.3 \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario4/KD2_structured

#### Run 4.11: KD3 + Structured (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd3-banglabert-to-sahajbert" \
    --prune_method structured \
    --prune_sparsity 0.3 \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario4/KD3_structured

#### Run 4.12: KD4 + Structured (eval_fold=2)

In [ ]:
!python main.py \
    --dataset_path "/kaggle/working/kd_pruning_quantization_framework_for_nlp/data/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-kd4-banglabert-to-banglabert-small" \
    --prune_method structured \
    --prune_sparsity 0.3 \
    --fine_tune_after_prune \
    --use_original_folds \
    --eval_fold 2 \
    --output_dir ./results/scenario4/KD4_structured

---
## Final Status Check
---

In [ ]:
import os
import json
from datetime import datetime

print(f"\n{'='*80}")
print(f"EXPERIMENT COMPLETION STATUS - {datetime.now()}")
print(f"{'='*80}\n")

experiments = [
    # Scenario 3: Magnitude on KD models
    ("3.3 KD1 + Magnitude", "./results/scenario3/KD1_magnitude", 3),
    ("3.4 KD2 + Magnitude", "./results/scenario3/KD2_magnitude", 3),
    ("3.5 KD3 + Magnitude", "./results/scenario3/KD3_magnitude", 2),
    ("3.6 KD4 + Magnitude", "./results/scenario3/KD4_magnitude", 2),
    # Scenario 4: Wanda
    ("4.1 KD1 + Wanda", "./results/scenario4/KD1_wanda", 3),
    ("4.2 KD2 + Wanda", "./results/scenario4/KD2_wanda", 3),
    ("4.3 KD3 + Wanda", "./results/scenario4/KD3_wanda", 2),
    ("4.4 KD4 + Wanda", "./results/scenario4/KD4_wanda", 2),
    # Scenario 4: Gradual
    ("4.5 KD1 + Gradual", "./results/scenario4/KD1_gradual", 3),
    ("4.6 KD2 + Gradual", "./results/scenario4/KD2_gradual", 3),
    ("4.7 KD3 + Gradual", "./results/scenario4/KD3_gradual", 2),
    ("4.8 KD4 + Gradual", "./results/scenario4/KD4_gradual", 2),
    # Scenario 4: Structured
    ("4.9 KD1 + Structured", "./results/scenario4/KD1_structured", 3),
    ("4.10 KD2 + Structured", "./results/scenario4/KD2_structured", 3),
    ("4.11 KD3 + Structured", "./results/scenario4/KD3_structured", 2),
    ("4.12 KD4 + Structured", "./results/scenario4/KD4_structured", 2),
]

success_count = 0

print(f"{'Experiment':<25} {'Fold':<6} {'F1 Weighted':<12} {'F1 Macro':<12} {'Status'}")
print("-" * 70)

for name, output_dir, fold in experiments:
    json_path = os.path.join(output_dir, "results_final.json")
    
    if os.path.exists(json_path):
        with open(json_path) as f:
            data = json.load(f)
        
        # Get final metrics
        if isinstance(data, list):
            final_metrics = data[-1]
        else:
            final_metrics = data
        
        f1_weighted = final_metrics.get('f1_weighted', 'N/A')
        f1_macro = final_metrics.get('f1_macro', 'N/A')
        
        print(f"{name:<25} {fold:<6} {f1_weighted:<12.4f} {f1_macro:<12.4f} SUCCESS")
        success_count += 1
    else:
        print(f"{name:<25} {fold:<6} {'N/A':<12} {'N/A':<12} FAILED")

print("-" * 70)
print(f"\nCompleted: {success_count}/{len(experiments)} experiments")
print(f"{'='*80}")

In [ ]:
# Copy results to output
!cp -r ./results /kaggle/working/
!python aggregate_results.py --results_dir ./results --output /kaggle/working/phase_cd_summary.csv --format all

In [ ]:
# Generate pruning methods comparison
import pandas as pd
import os
import json

# Collect all results
comparison_data = []

methods = ['magnitude', 'wanda', 'gradual', 'structured']
kd_info = [
    ('KD1', 3),  # KD1 uses T1's fold
    ('KD2', 3),  # KD2 uses T1's fold
    ('KD3', 2),  # KD3 uses T2's fold
    ('KD4', 2),  # KD4 uses T2's fold
]

for method in methods:
    for kd_id, fold in kd_info:
        if method == 'magnitude':
            path = f"./results/scenario3/{kd_id}_magnitude/results_final.json"
        else:
            path = f"./results/scenario4/{kd_id}_{method}/results_final.json"
        
        if os.path.exists(path):
            with open(path) as f:
                data = json.load(f)
            
            # Get final metrics
            if isinstance(data, list):
                final_metrics = data[-1]
            else:
                final_metrics = data
            
            comparison_data.append({
                'KD_Model': kd_id,
                'Fold': fold,
                'Prune_Method': method,
                'F1_Weighted': final_metrics.get('f1_weighted', None),
                'F1_Macro': final_metrics.get('f1_macro', None),
                'Accuracy': final_metrics.get('accuracy', None),
            })

if comparison_data:
    df = pd.DataFrame(comparison_data)
    print("\n" + "="*70)
    print("PRUNING METHODS COMPARISON (F1 Weighted)")
    print("="*70 + "\n")
    
    # Pivot table for easy comparison
    pivot = df.pivot(index='KD_Model', columns='Prune_Method', values='F1_Weighted')
    print(pivot.to_string())
    
    # Save comparison
    pivot.to_csv('/kaggle/working/pruning_methods_comparison.csv')
    print("\nSaved to /kaggle/working/pruning_methods_comparison.csv")
else:
    print("No results available for comparison yet.")

In [ ]:
print(f"\n{'='*60}")
print("PHASE C+D COMPLETE!")
print(f"{'='*60}")
print(f"\nCompleted at: {datetime.now()}")
print("\nAll experiments done! Combine results from Phase A, B, C+D for final analysis.")